In [0]:
%sql
CREATE OR REPLACE TABLE datalakehouse.silver.tbl_salario_serie_1619 (
  ID INT,
  DATA_REFERENCIA DATE,
  NUM_MES INT,
  ANO INT,
  VALOR DECIMAL(10,2),
  MOEDA VARCHAR(4)
);

In [0]:
df = spark.sql(
f'''
SELECT  
  to_date(data, 'dd/MM/yyyy') AS DATA_REFERENCIA,
  month(to_date(data, 'dd/MM/yyyy')) AS NUM_MES,
  year(to_date(data, 'dd/MM/yyyy')) AS ANO,
  CAST(valor AS DECIMAL(10,2)) AS VALOR,
  CASE 
    WHEN data = '01/02/1990' THEN 'NCz$'
    WHEN data = '01/03/1990' THEN 'NCz$'
    WHEN data >= '01/04/1990' AND data <= '01/08/1993' THEN 'Cr$'
    WHEN data >= '01/09/1993' AND data <= '01/02/1994' THEN 'CR$'
    WHEN data >= '01/03/1994' THEN 'R$'
  END AS MOEDA
FROM datalakehouse.bronze.bcb_salario_serie_1619 '''
)

df.createOrReplaceTempView("tbl_salario_serie_1619")

In [0]:
%sql
TRUNCATE TABLE datalakehouse.silver.tbl_salario_serie_1619;

INSERT INTO datalakehouse.silver.tbl_salario_serie_1619
SELECT  
  row_number() OVER (ORDER BY ANO) AS ID,
  DATA_REFERENCIA,
  NUM_MES,
  ANO,
  VALOR,
  MOEDA
FROM tbl_salario_serie_1619;